# Bunu: a health check for your DataFrame

Observe first. Mutate never by surprise.  
`pip install bunu`

In [1]:
import bunu

df = bunu.demo()   # a deliberately messy dataset, no files needed
df.head()

,customer_id,email,age,revenue,country,signup_date,phone_note,plan
0,0,user0@example.com,unknown,76.49,germany,2024-03-05,ok,free
1,1,user1@example.com,unknown,99.60,Germany,2024-03-05,ok,free
2,2,user2@example.com,unknown,107.45,India,2024-03-05,ok,free
3,3,user3@example.com,unknown,inf,UK,2024-03-05,ok,free
4,4,user4@example.com,unknown,97.87,Germany,2024-03-05,ok,free


## 1. Is this data safe to use?

In [2]:
bunu.check(df)

HIGH,emailDUPLICATE_KEY,"99.4% unique, but 12 value(s) repeat; expected to be a unique key. (12)"
HIGH,revenueINFINITE_VALUES,"Contains inf / -inf, which break means, scalers and models. (1)suggested fix (not applied)df['revenue'] = df['revenue'].replace([float('inf'), float('-inf')], float('nan'))"
HIGH,signup_dateDATETIME_MIXED_FORMAT,"Dates are written in 2 different formats (like '2024-03-05': 1,867, like '05/03/2024': 133); parsing them together can silently swap day and month or produce NaT. (133)suggested fix (not applied)# parse each format separately with an explicit format=, then combine"
MEDIUM,ageNUMERIC_AS_STRING,"98.8% of values are numbers stored as text; 25 are not numeric (e.g. 'unknown'). (1,975)suggested fix (not applied)df['age'] = pd.to_numeric(df['age'], errors='coerce') # 25 value(s) will become NaN"
MEDIUM,phone_noteNULL_TOKENS,"Text that looks like missing data ('N/A'). (479)suggested fix (not applied)df['phone_note'] = df['phone_note'].replace(['N/A'], pd.NA)"
LOW,countryCASE_VARIANTS,"4 value(s) appear in several spellings that differ only by case or spaces, e.g. 'germany' / 'Germany'; 'India' / ' India' / 'india'; 'UK' / 'uk'. Nothing was changed. (22)suggested fix (not applied)df['country'] = df['country'].str.strip().str.casefold() # only if these really mean the same thing"
LOW,countryRARE_CATEGORY,1 rare value(s) in a column with 8 common ones (e.g. ' India'); possible typos. (1)
LOW,countryWHITESPACE_PADDED,Values have leading or trailing spaces. (1)suggested fix (not applied)df['country'] = df['country'].str.strip()
LOW,planCONSTANT_COLUMN,"Only one distinct value ('free'); carries no information. (2,000)"
LOW,revenueEXTREME_VALUES,1 value(s) lie more than 5 IQR outside the middle 50% (normal range ~ -15.39 to 214.6). (1)
LOW,revenueMISSING_VALUES,8.1% of values are missing. (162)


## 2. What should I look at first?

In [3]:
bunu.explain(df)

DataFrame
------------------------------------------------------------
2,000 rows x 8 columns

Healthy             1 column(s)
Needs attention     4 column(s)
Suspicious          3 column(s)

Most important
  1. [HIGH] email: 99.4% unique, but 12 value(s) repeat; expected to be a unique key.
  2. [HIGH] revenue: Contains inf / -inf, which break means, scalers and models.
  3. [HIGH] signup_date: Dates are written in 2 different formats (like '2024-03-05': 1,867, like '05/03/2024': 133); parsing them together can silently swap day and month or produce NaT.
  4. [MEDIUM] age: 98.8% of values are numbers stored as text; 25 are not numeric (e.g. 'unknown').
  5. [MEDIUM] phone_note: Text that looks like missing data ('N/A').

Suggested next step (not applied)
  df['revenue'] = df['revenue'].replace([float('inf'), float('-inf')], float('nan'))
------------------------------------------------------------
Bunu never modifies your data.

## 3. Pipeline broke overnight. What changed?

In [4]:
yesterday, today = bunu.demo(pair=True)
bunu.compare(yesterday, today)

HIGH,customer_idDUPLICATES_GAINED,"Was 100% unique, now has 530 duplicate value(s). (530)"
HIGH,revenueMEAN_SHIFT,Mean moved 99.78 -> 226.9 (8.4 old std devs).
HIGH,signup_dateNULL_RATE_CHANGED,Missing values: 0.0% -> 44.0%.
MEDIUM,(dataset)ROW_COUNT_CHANGED,"Rows changed by +25.0% (2,000 -> 2,500). (500)"
MEDIUM,channelSCHEMA_COLUMN_ADDED,New column appeared.
LOW,countryCATEGORIES_GONE,1 value(s) no longer present: ' India'
LOW,countryNEW_CATEGORIES,1 new value(s): 'France'


## 4. Keep a few KB instead of yesterday's data

In [5]:
bunu.snapshot(yesterday).save('yesterday.json')
bunu.compare('yesterday.json', today).get(min_severity='high')

[Issue(code='DUPLICATES_GAINED', severity=<Severity.HIGH: 3>, message='Was 100% unique, now has 530 duplicate value(s).', column='customer_id', affected_rows=530, suggestion=None, details={}),
 Issue(code='MEAN_SHIFT', severity=<Severity.HIGH: 3>, message='Mean moved 99.78 -> 226.9 (8.4 old std devs).', column='revenue', affected_rows=None, suggestion=None, details={'z': 8.37258678341317}),
 Issue(code='NULL_RATE_CHANGED', severity=<Severity.HIGH: 3>, message='Missing values: 0.0% -> 44.0%.', column='signup_date', affected_rows=None, suggestion=None, details={'old': 0.0, 'new': 0.4404})]

## 5. Stop a pipeline on bad data

In [6]:
try:
    bunu.check(df).raise_if('high')
except bunu.DataQualityError as e:
    print(e)

Bunu found 3 issue(s) at or above 'high':
  - HIGH DUPLICATE_KEY [email] 99.4% unique, but 12 value(s) repeat; expected to be a unique key.
  - HIGH INFINITE_VALUES [revenue] Contains inf / -inf, which break means, scalers and models.
  - HIGH DATETIME_MIXED_FORMAT [signup_date] Dates are written in 2 different formats (like '2024-03-05': 1,867, like '05/03/2024': 133); parsing them together can silently swap day and month or produce NaT.
